<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-02-reliable-tools-for-orbit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 (graded) — Reliable tools for Orbit
**Course 3: AI Agents and Agentic AI with Python — Chapter 2: Tools & function calling**

**Problem brief (Leo Farkas, Orbit Retail):** "The returns assistant needs to actually
query our inventory database and compute refunds — correctly, every time. It once refunded
a customer 400% because it misread a tool result."

**What you'll submit:** a real SQL tool, a refund calculator, and a unit converter — with
schema validation, 10 rejected malformed calls, and a tool-call-accuracy scorecard on 20 tasks.

## 1. Build a real orders database (from Online Retail II, with offline fallback)

In [ ]:
import sqlite3
import io
import urllib.request
import pandas as pd
import numpy as np

np.random.seed(0)

def load_orders():
    try:
        # pd.read_excel(url) has no timeout and can hang indefinitely on a stalled connection;
        # fetching the bytes ourselves with a bounded timeout lets a real network problem raise
        # promptly and land in the offline fallback below, instead of hanging the whole cell.
        req = urllib.request.Request(
            'https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx',
            headers={'User-Agent': 'aibits-course-lab/1.0'},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read()
        df = pd.read_excel(io.BytesIO(raw), sheet_name=0, nrows=2000)
        df = df.rename(columns={'Invoice': 'order_id', 'StockCode': 'sku', 'Description': 'item',
                                  'Quantity': 'quantity', 'Price': 'unit_price', 'Customer ID': 'customer_id'})
        df = df.dropna(subset=['order_id', 'customer_id'])
        df['order_id'] = df['order_id'].astype(str)
        df['customer_id'] = df['customer_id'].astype(int)
        print(f'Loaded {len(df)} real Online Retail II order lines.')
        return df[['order_id', 'customer_id', 'sku', 'item', 'quantity', 'unit_price']]
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — a small synthetic orders table.')
        n = 500
        return pd.DataFrame({
            'order_id': [f'ORD{i:05d}' for i in range(n)],
            'customer_id': np.random.randint(1000, 1050, n),
            'sku': [f'SKU{np.random.randint(100, 130)}' for _ in range(n)],
            'item': np.random.choice(['Mug', 'Notebook', 'Lamp', 'Cushion', 'Candle'], n),
            'quantity': np.random.randint(1, 6, n),
            'unit_price': np.round(np.random.uniform(3, 45, n), 2),
        })

orders_df = load_orders()
conn = sqlite3.connect(':memory:')
orders_df.to_sql('orders', conn, index=False, if_exists='replace')
print(pd.read_sql('SELECT * FROM orders LIMIT 3', conn))

## 2. The tools, with validation

In [ ]:
import re

def tool_query_order(order_id: str) -> dict:
    """Read-only, parameterized lookup — never string-formats user input into SQL."""
    if not re.match(r'^[A-Za-z0-9]{1,20}$', str(order_id)):
        return {'error': f'invalid order_id format: {order_id!r}'}
    rows = pd.read_sql('SELECT * FROM orders WHERE order_id = ?', conn, params=(str(order_id),))
    if rows.empty:
        return {'error': f'no such order: {order_id}'}
    total = float((rows['quantity'] * rows['unit_price']).sum())
    return {'order_id': order_id, 'items': rows['item'].tolist(), 'order_total_usd': round(total, 2)}


REFUND_RATES = {'damaged': 1.0, 'wrong_item': 1.0, 'changed_mind': 0.85}  # a restocking fee for changed_mind

def tool_compute_refund(order_id: str, reason: str) -> dict:
    if reason not in REFUND_RATES:
        return {'error': f'invalid reason {reason!r}; must be one of {list(REFUND_RATES)}'}
    order = tool_query_order(order_id)
    if 'error' in order:
        return order
    # bounded explicitly: the refund can NEVER exceed the order total — this is exactly what
    # would have stopped Orbit's 400% bug
    refund = round(min(order['order_total_usd'], order['order_total_usd'] * REFUND_RATES[reason]), 2)
    return {'order_id': order_id, 'refund_amount_usd': refund}


UNIT_FACTORS = {('kg', 'lb'): 2.20462, ('lb', 'kg'): 1 / 2.20462, ('cm', 'in'): 0.393701, ('in', 'cm'): 2.54}

def tool_convert_unit(value: float, from_unit: str, to_unit: str) -> dict:
    if not isinstance(value, (int, float)):
        return {'error': f'value must be numeric, got {type(value).__name__}'}
    key = (from_unit, to_unit)
    if key not in UNIT_FACTORS:
        return {'error': f'unsupported conversion: {from_unit} -> {to_unit}'}
    return {'result': round(value * UNIT_FACTORS[key], 3), 'unit': to_unit}

TOOLS = {'query_order': tool_query_order, 'compute_refund': tool_compute_refund, 'convert_unit': tool_convert_unit}

## 3. Reject 10 malformed calls

In [ ]:
malformed_calls = [
    ('query_order', {'order_id': "'; DROP TABLE orders; --"}),
    ('query_order', {'order_id': ''}),
    ('query_order', {'order_id': 'x' * 50}),
    ('compute_refund', {'order_id': orders_df.iloc[0]['order_id'], 'reason': 'i felt like it'}),
    ('compute_refund', {'order_id': 'NOTREAL', 'reason': 'damaged'}),
    ('convert_unit', {'value': 'ten', 'from_unit': 'kg', 'to_unit': 'lb'}),
    ('convert_unit', {'value': 5, 'from_unit': 'furlongs', 'to_unit': 'lb'}),
    ('convert_unit', {'value': 5, 'from_unit': 'kg', 'to_unit': 'parsecs'}),
    ('query_order', {'order_id': None}),
    ('compute_refund', {'order_id': orders_df.iloc[0]['order_id'], 'reason': None}),
]

rejected = 0
for tool_name, args in malformed_calls:
    try:
        result = TOOLS[tool_name](**args)
        ok = 'error' in result
    except Exception as e:
        result = {'error': str(e)}
        ok = True
    print(f"[{'rejected' if ok else 'NOT REJECTED — BUG'}] {tool_name}({args}) -> {result}")
    rejected += int(ok)

print(f'\n{rejected}/{len(malformed_calls)} malformed calls correctly rejected.')
assert rejected == len(malformed_calls), 'Every malformed call should be rejected.'

## 4. Tool-call-accuracy scorecard on 20 valid tasks

In [ ]:
sample_order_ids = orders_df['order_id'].drop_duplicates().sample(10, random_state=0).tolist()
tasks = (
    [('query_order', {'order_id': oid}) for oid in sample_order_ids] +
    [('compute_refund', {'order_id': oid, 'reason': 'damaged'}) for oid in sample_order_ids[:5]] +
    [('convert_unit', {'value': 2.5, 'from_unit': 'kg', 'to_unit': 'lb'})] * 5
)

correct = 0
for tool_name, args in tasks:
    result = TOOLS[tool_name](**args)
    ok = 'error' not in result
    correct += int(ok)

print(f'Tool-call accuracy: {correct}/{len(tasks)} = {correct/len(tasks):.0%}')

## 5. Write-up (fill in)
Point to the exact line in `tool_compute_refund` that structurally prevents Orbit's original
400%-refund bug from recurring — a refund tool that returns an unbounded percentage instead
of a bounded dollar amount is how that bug happened in the first place.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 2: Tools & function calling*